# Overview

1. First, upload the `movies_data_1.csv` file as a table into Databricks. To do that, click **New → Add or upload data → Create or modify a table**, then upload the CSV file.
2. After the file is uploaded, set the **Table Name** and open **Advanced attributes**. Make sure the following options are checked: `First row contains the header`, `Automatically detect column types`, and `Rows span multiple lines` so that the file is properly parsed.
3. Click the **Create table** button which will redirect you to a new page where you'll see the schema and sample data.
4. Next, read the table as a pandas dataframe.

In [0]:
# Read table into spark dataframe and convert it to pandas
movies = spark.read.table("dbacademy.default.movies_data").toPandas()

movies.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,Extract_date,owner_company,nr_of_episodes
0,Blood Red Sky,(2021),"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,None,2020-01-01 00:00:00.000000,Columbia Pictures\t,109
1,None,None,None,NaN,None,None,None,NaN,None,2020-01-01 00:25:41.336487,20th Century Fox\t,75
2,None,None,None,NaN,None,None,None,NaN,None,2020-01-01 00:51:22.672974,Marvel Studios,72
3,None,None,None,NaN,None,None,None,NaN,None,2020-01-01 01:17:04.009461,Universal Pictures\t,118
4,None,None,None,NaN,None,None,None,NaN,None,2020-01-01 01:42:45.345949,Columbia Pictures\t,142


### Movies dataset description

*  **Movie:** movie name
*  **Year:** release year
*  **Genre:** movie genre
*  **Rating:** audience rating
*  **One-Line:** short description about the movie
*  **Stars:** the casting art. Contains the director and star actors
*  **Votes:** audience votes
*  **Runtime:** Duration of movie
*  **Gross:** total amount earned worldwide
*  **Extract_date:** datetime of extraction
*  **owner_company:** Owner company of the movie/tvshow
*  **nr_of_episodes:** number of episodes


#### Requirement: Data cleaning

Apply the following steps for data cleaning:
1. Clean dataset by removing empty rows. An empty row is considered when column Movies,Year, Genre, Rating, One-Line,Stars, Votes, RunTime and Gross are all null.
2. Clean whitespaces, remove the '\n' from columns Genre, One-Line and Stars and '\t' from owner_company


In [0]:

# Data cleaning requirements
# write your code here

# Step 1: Remove completely empty rows
cols_to_check = ['MOVIES', 'YEAR', 'GENRE', 'RATING', 'ONE-LINE', 
                 'STARS', 'VOTES', 'RunTime', 'Gross']
movies = movies.dropna(subset=cols_to_check, how='all')

# Step 2: Clean newline characters from text columns
movies['GENRE'] = movies['GENRE'].str.replace('\n', ' ', regex=False).str.strip()
movies['ONE-LINE'] = movies['ONE-LINE'].str.replace('\n', ' ', regex=False).str.strip()
movies['STARS'] = movies['STARS'].str.replace('\n', ' ', regex=False).str.strip()

# Step 3: Clean tab characters from owner_company
movies['owner_company'] = movies['owner_company'].str.replace('\t', '', regex=False).str.strip()

movies

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,Extract_date,owner_company,nr_of_episodes
0,Blood Red Sky,(2021),"Action, Horror, Thriller",6.1,A woman with a mysterious illness is forced in...,Director: Peter Thorwarth | Stars: Peri B...,"21,062",121.0,None,2020-01-01 00:00:00.000000,Columbia Pictures,109
10,Masters of the Universe: Revelation,(2021– ),"Animation, Action, Adventure",5.0,The war for Eternia begins again in what may b...,"Stars: Chris Wood, Sarah Michelle Gellar, Le...","17,870",25.0,None,2020-01-01 04:16:53.364872,Columbia Pictures,20
11,The Walking Dead,(2010–2022),"Drama, Horror, Thriller",8.2,Sheriff Deputy Rick Grimes wakes up from a com...,"Stars: Andrew Lincoln, Norman Reedus, Meliss...","885,805",44.0,None,2020-01-01 04:42:34.701360,Legendary Entertainment,80
12,Rick and Morty,(2013– ),"Animation, Adventure, Comedy",9.2,An animated series that follows the exploits o...,"Stars: Justin Roiland, Chris Parnell, Spence...","414,849",23.0,None,2020-01-01 05:08:16.037847,Universal Pictures,14
13,Army of Thieves,(2021),"Action, Crime, Horror",NaN,"A prequel, set before the events of Army of th...",Director: Matthias Schweighöfer | Stars: ...,None,NaN,None,2020-01-01 05:33:57.374334,Paramount Pictures,18
...,...,...,...,...,...,...,...,...,...,...,...,...
10142,The Imperfects,(2021– ),"Adventure, Drama, Fantasy",NaN,Add a Plot,"Stars: Morgan Taylor Campbell, Chris Cope, I...",None,NaN,None,2020-06-29 22:17:14.654050,Warner Bros.,115
10143,Arcane,(2021– ),"Animation, Action, Adventure",NaN,Add a Plot,,None,NaN,None,2020-06-29 22:42:55.990538,Paramount Pictures,82
10144,Heart of Invictus,(2022– ),"Documentary, Sport",NaN,Add a Plot,Director: Orlando von Einsiedel | Star: P...,None,NaN,None,2020-06-29 23:08:37.327025,Relativity Media,39
10145,The Imperfects,(2021– ),"Adventure, Drama, Fantasy",NaN,Add a Plot,Director: Jovanka Vuckovic | Stars: Morga...,None,NaN,None,2020-06-29 23:34:18.663512,Relativity Media,60


#### Requirement: Column splitting

Apply the following steps for extracting data into separate columns:

3. Create separate columns _"Director"_ and _"Stars"_ from column "STARS". Extract the director name and store it into the newly created column _"Director"_. Do the same thing for stars. In the end, drop the original "STARS" column.

4. Store separately date and timestamp from Extract_date column as new columns <code>extraction_date</code> and <code>extraction_time</code>. Drop the original "Extract_date" column in the end.



In [0]:

# write your code here

import re

def extract_director(stars_string):
    if pd.isna(stars_string):
        return None
    match = re.search(r'Director:\s*(.+?)(?:\s*\||\s*Stars:|$)', str(stars_string), re.IGNORECASE)
    return match.group(1).strip() if match else None

def extract_stars(stars_string):
    if pd.isna(stars_string):
        return None
    match = re.search(r'Stars:\s*(.+?)(?:\s*\||\s*Director:|$)', str(stars_string), re.IGNORECASE)
    return match.group(1).strip() if match else None

# Extract Director and Stars from the STARS column
movies['Director'] = movies['STARS'].apply(extract_director)
movies['Stars'] = movies['STARS'].apply(extract_stars)

# Drop original STARS column
movies.drop(columns=['STARS'], inplace=True)

# Extract date and time components from Extract_date
movies['extraction_date'] = movies['Extract_date'].dt.date
movies['extraction_time'] = movies['Extract_date'].dt.time

# Drop original Extract_date column
movies.drop(columns=['Extract_date'], inplace=True)






#### Requirement: Year logic

Apply the following steps for extracting data into separate columns:


5. Calculate how long did the movie/tv show last and store the value in a separate column **"lasted"**. Also, store in new columns the **start_year** and **end_year** values of the movie when applicable. If the movie is still in production, then fill end_year with value <code>'present'</code>. In the end, drop the original "YEAR" column from the dataset.


In [0]:
# Write your code here
import numpy as np
import pandas as pd

movies['start_year'] = movies['YEAR'].str.extract(r'\((\d{4})', expand=False)

movies['end_year'] = movies['YEAR'].str.extract(r'[–\-]\s*(\d{4})', expand=False)

has_dash = movies['YEAR'].str.contains(r'[–\-]', na=False)
year_exists = movies['YEAR'].notna()

movies['end_year'] = np.where(
    movies['end_year'].notna(),
    movies['end_year'],
    np.where(
        has_dash & year_exists,
        'present',
        movies['start_year']
    )
)


def compute_lasted(row):
    if pd.isna(row['start_year']):
        return ''
    if row['start_year'] == row['end_year']:
        return ''
    if row['end_year'] == 'present':
        return 'present'
    if pd.notna(row['end_year']):
        return int(float(row['end_year'])) - int(float(row['start_year']))
    return ''

movies['lasted'] = movies.apply(compute_lasted, axis=1)

movies.drop(columns=['YEAR'], inplace=True)

movies.loc[movies["lasted"]=='']['lasted']



0         
13        
34        
38        
40        
        ..
10113     
10114     
10115     
10116     
10117     
Name: lasted, Length: 5431, dtype: object

#### Requirement: Dimension dataframes

Apply the following steps for extracting data into separate columns:


6. Extract unique values from **owner_company** column and store them as separate pandas Dataframe called <code>DimCompany</code>
7. Similarly, extract unique values from **director column** and store it as separate pandas Dataframe called <code>DimDirector</code>




In [0]:
# Write your code here
import pandas as pd
DimCompany = pd.DataFrame(movies.owner_company.unique())

DimCompany.columns = ['Owner Company']

DimCompany.head()

DimDirector = pd.DataFrame(movies.Director.unique())

DimDirector.columns = ['Director']

movies




,MOVIES,GENRE,RATING,ONE-LINE,VOTES,RunTime,Gross,owner_company,nr_of_episodes,Director,Stars,extraction_date,extraction_time,start_year,end_year,lasted
0,Blood Red Sky,"Action, Horror, Thriller",6.1,A woman with a mysterious illness is forced in...,"21,062",121.0,None,Columbia Pictures,109,Peter Thorwarth,"Peri Baumeister, Carl Anton Koch, Alexander ...",2020-01-01,00:00:00,2021,2021,
10,Masters of the Universe: Revelation,"Animation, Action, Adventure",5.0,The war for Eternia begins again in what may b...,"17,870",25.0,None,Columbia Pictures,20,None,"Chris Wood, Sarah Michelle Gellar, Lena Head...",2020-01-01,04:16:53.364872,2021,present,present
11,The Walking Dead,"Drama, Horror, Thriller",8.2,Sheriff Deputy Rick Grimes wakes up from a com...,"885,805",44.0,None,Legendary Entertainment,80,None,"Andrew Lincoln, Norman Reedus, Melissa McBri...",2020-01-01,04:42:34.701360,2010,2022,12
12,Rick and Morty,"Animation, Adventure, Comedy",9.2,An animated series that follows the exploits o...,"414,849",23.0,None,Universal Pictures,14,None,"Justin Roiland, Chris Parnell, Spencer Gramm...",2020-01-01,05:08:16.037847,2013,present,present
13,Army of Thieves,"Action, Crime, Horror",NaN,"A prequel, set before the events of Army of th...",None,NaN,None,Paramount Pictures,18,Matthias Schweighöfer,"Matthias Schweighöfer, Nathalie Emmanuel, Ru...",2020-01-01,05:33:57.374334,2021,2021,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10142,The Imperfects,"Adventure, Drama, Fantasy",NaN,Add a Plot,None,NaN,None,Warner Bros.,115,None,"Morgan Taylor Campbell, Chris Cope, Iñaki Go...",2020-06-29,22:17:14.654050,2021,present,present
10143,Arcane,"Animation, Action, Adventure",NaN,Add a Plot,None,NaN,None,Paramount Pictures,82,None,None,2020-06-29,22:42:55.990538,2021,present,present
10144,Heart of Invictus,"Documentary, Sport",NaN,Add a Plot,None,NaN,None,Relativity Media,39,Orlando von Einsiedel,None,2020-06-29,23:08:37.327025,2022,present,present
10145,The Imperfects,"Adventure, Drama, Fantasy",NaN,Add a Plot,None,NaN,None,Relativity Media,60,Jovanka Vuckovic,"Morgan Taylor Campbell, Iñaki Godoy, Rhianna...",2020-06-29,23:34:18.663512,2021,present,present


## Assertions

The cells below are assertions for checking  whether the requirements were fulfilled on the movies dataframe. You can execute the cell to check your work. If everything passes, then you have successfully implemented the requirements.

In [0]:
import numpy as np

def assert_df(df):

    # Checks for the first csv file
    # the end dataset should be of shape (9999,16)
    assert df.shape[0] == 9999
    assert df.shape[1] == 16

    # The Lasted column should have 3180 rows with 'present' value
    assert df.loc[df["end_year"]=='present'].shape[0] == 3180
    #assert df.loc[df["lasted"]==''].shape[0] == 8611 


    # random checks of data
    assert df.iloc[0]['Director'] == 'Peter Thorwarth'
    assert df.iloc[11]['MOVIES'] == 'Lucifer'
    assert df.iloc[11]['lasted'] == 5

    # there should be 10 unique values for companies
    assert len(df.owner_company.unique()) == 10

    # check if the unique values for companies are the same
    np.testing.assert_array_equal(df.owner_company.unique(),['Columbia Pictures', 'Legendary Entertainment',
        'Universal Pictures', 'Paramount Pictures', 'Walt Disney Pictures',
        'Marvel Studios', '20th Century Fox', 'Relativity Media',
        'RatPac-Dune Entertainment', 'Warner Bros.'])

    # there should be 3140 unique values for directors (excluding NaN)
    assert len(df.Director.dropna().unique()) == 3140
    print("Assertion of dataframe is complete!")


In [0]:
# Pass in the name of your movies dataframe
assert_df(movies)


Assertion of dataframe is complete!
